In [ ]:
import pandas as pd
import os 


In [11]:
os.makedirs("data/structured_files", exist_ok=True)

In [12]:
data = {
    "Book ID": [101, 102, 103, 104],
    "Title": ["Atomic Habits", "Deep Work", "Ikigai", "The Alchemist"],
    "Author": ["James Clear", "Cal Newport", "Hector Garcia", "Paulo Coelho"],
    "Rating": [4.8, 4.7, 4.5, 4.6],
    "Price": [499, 399, 299, 350],
    "Description": [
        "A guide to building good habits and breaking bad ones.",
        "Focuses on deep, meaningful work in a distracted world.",
        "Explores the Japanese concept of purpose and happiness.",
        "A philosophical story about following your dreams."
    ]
}

df=pd.DataFrame(data)
df.to_csv('data/structured_files/books.csv')

In [ ]:
import pandas as pd

users_data = {
    "User ID": [1, 2, 3, 4],
    "User Name": ["Mayank", "Amit", "Sara", "Riya"],
    "Book ID": [101, 102, 101, 104],
    "Review": [
        "Amazing book!",
        "Very useful for productivity",
        "Changed my habits",
        "Inspirational story"
    ],
    "Rating Given": [5, 4, 5, 4]
}

with pd.ExcelWriter('data/structured_files/inventory.xlsx') as writer:
    df.to_excel(writer, sheet_name='Books', index=False)
    pd.DataFrame(users_data).to_excel(writer, sheet_name='Users', index=False)

In [20]:
from langchain_community.document_loaders import CSVLoader
from langchain_community.document_loaders import UnstructuredCSVLoader

print("CSV Loader")
csv_loader=CSVLoader(
    file_path='data/structured_files/books.csv',
    encoding='utf-8',
    csv_args={
        'delimiter':',',
        'quotechar':'"',
    }
)

csv_docs=csv_loader.load()
print(csv_docs)

CSV Loader
[Document(metadata={'source': 'data/structured_files/books.csv', 'row': 0}, page_content=': 0\nBook ID: 101\nTitle: Atomic Habits\nAuthor: James Clear\nRating: 4.8\nPrice: 499\nDescription: A guide to building good habits and breaking bad ones.'), Document(metadata={'source': 'data/structured_files/books.csv', 'row': 1}, page_content=': 1\nBook ID: 102\nTitle: Deep Work\nAuthor: Cal Newport\nRating: 4.7\nPrice: 399\nDescription: Focuses on deep, meaningful work in a distracted world.'), Document(metadata={'source': 'data/structured_files/books.csv', 'row': 2}, page_content=': 2\nBook ID: 103\nTitle: Ikigai\nAuthor: Hector Garcia\nRating: 4.5\nPrice: 299\nDescription: Explores the Japanese concept of purpose and happiness.'), Document(metadata={'source': 'data/structured_files/books.csv', 'row': 3}, page_content=': 3\nBook ID: 104\nTitle: The Alchemist\nAuthor: Paulo Coelho\nRating: 4.6\nPrice: 350\nDescription: A philosophical story about following your dreams.')]


In [37]:
from typing import List
from langchain_core.documents import Document
from numpy import append

print('custom csv processing ')
def process_csv_intelligently(filepath: str)-> List[Document]:
    """ process csv with intelligent document creation """
    df=pd.read_csv(filepath)
    documents=[]

    for idx,row in df.iterrows():
        content=f"""Book Information:
        Name: {row['Title']}
        Author: {row['Author']}
        Rating: {row['Rating']}
        Price: ₹{row['Price']}
        Description: {row['Description']}"""

        doc=Document(page_content=content,
                     metadata={
                         "source":filepath,
                         "row_index":idx,
                         "book_name":row['Title'],
                         "price":row['Price'],
                         "data_type":'book_info'
                     }
            )
        documents.append(doc)
    return documents

    

custom csv processing 


In [38]:
process_csv_intelligently('data/structured_files/books.csv')

[Document(metadata={'source': 'data/structured_files/books.csv', 'row_index': 0, 'book_name': 'Atomic Habits', 'price': 499, 'data_type': 'book_info'}, page_content='Book Information:\n        Name: Atomic Habits\n        Author: James Clear\n        Rating: 4.8\n        Price: ₹499\n        Description: A guide to building good habits and breaking bad ones.'),
 Document(metadata={'source': 'data/structured_files/books.csv', 'row_index': 1, 'book_name': 'Deep Work', 'price': 399, 'data_type': 'book_info'}, page_content='Book Information:\n        Name: Deep Work\n        Author: Cal Newport\n        Rating: 4.7\n        Price: ₹399\n        Description: Focuses on deep, meaningful work in a distracted world.'),
 Document(metadata={'source': 'data/structured_files/books.csv', 'row_index': 2, 'book_name': 'Ikigai', 'price': 299, 'data_type': 'book_info'}, page_content='Book Information:\n        Name: Ikigai\n        Author: Hector Garcia\n        Rating: 4.5\n        Price: ₹299\n      

In [40]:
print("EXCEL PROCESSING")
print("pandas based excel processing")

def process_excel_with_pandas(fielpath:str) -> List[Document]:
    """ process excel with pandas """
    xls=pd.ExcelFile(fielpath)
    documents=[]

    for sheet_name in xls.sheet_names:
        df=xls.parse(sheet_name)

        for idx,row in df.iterrows():
            content=f"""Sheet: {sheet_name}
            Row Index: {idx}
            Data: {row.to_dict()}"""

            doc=Document(page_content=content,
                         metadata={
                             "source":fielpath,
                             "sheet_name":sheet_name,
                             "row_index":idx,
                             "data_type":'excel_data'
                         }
                )
            documents.append(doc)
    return documents


EXCEL PROCESSING
pandas based excel processing


In [41]:
process_excel_with_pandas("data/structured_files/inventory.xlsx")

[Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Books', 'row_index': 0, 'data_type': 'excel_data'}, page_content="Sheet: Books\n            Row Index: 0\n            Data: {'Book ID': 101, 'Title': 'Atomic Habits', 'Author': 'James Clear', 'Rating': 4.8, 'Price': 499, 'Description': 'A guide to building good habits and breaking bad ones.'}"),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Books', 'row_index': 1, 'data_type': 'excel_data'}, page_content="Sheet: Books\n            Row Index: 1\n            Data: {'Book ID': 102, 'Title': 'Deep Work', 'Author': 'Cal Newport', 'Rating': 4.7, 'Price': 399, 'Description': 'Focuses on deep, meaningful work in a distracted world.'}"),
 Document(metadata={'source': 'data/structured_files/inventory.xlsx', 'sheet_name': 'Books', 'row_index': 2, 'data_type': 'excel_data'}, page_content="Sheet: Books\n            Row Index: 2\n            Data: {'Book ID': 103, 'Title': 'Iki